# ForestWatch AI — satellite pipeline

Neural Storm 2026 · GP-04 · Working notebook

This notebook verifies Earth Engine access and checks Sentinel-2 data for a pilot area. A satellite-observed change is only a candidate for human review; it is not proof of illegal deforestation.

**Run the two code cells from top to bottom. Do not skip the setup cell.**

## 1. Install required package

Run this cell first. Wait until it finishes.

In [ ]:
%pip -q install earthengine-api

## 2. Authenticate, initialize and check satellite data

Run this entire cell after Step 1. If Google asks you to authorize Earth Engine, follow the on-screen prompts.

In [ ]:
import ee

PROJECT_ID = 'forestwatch-ai-2026'
START_DATE = '2025-01-01'
END_DATE = '2025-12-31'

print('Authenticating with Google Earth Engine...')
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print('Earth Engine initialized successfully.')

# Pilot test area in the central Western Ghats.
# This rectangle is only a data-access test area, not a final study boundary.
aoi = ee.Geometry.Rectangle([74.5, 12.0, 75.5, 14.0])

sentinel2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
)

scene_count = sentinel2.size().getInfo()
print('\n--- ForestWatch data access check ---')
print(f'Earth Engine project: {PROJECT_ID}')
print(f'Date window: {START_DATE} to {END_DATE}')
print(f'Sentinel-2 scenes found: {scene_count}')
print('Result: Sentinel-2 collection is accessible.' if scene_count > 0 else 'Result: No scenes found for this area and date window.')

## 3. First vegetation-change screening (pilot)

This cell builds cloud-screened Sentinel-2 NDVI composites for the same Jan–Mar season in 2024 and 2025, then maps pixels with a large NDVI decrease. This is an exploratory vegetation-change screen only—not a deforestation classifier or proof of illegal activity. The pilot box is deliberately smaller than the initial data-access rectangle to keep processing manageable.

In [ ]:
# Re-initialize safely if this cell is run after a Colab runtime reset.
import ee
from IPython.display import display, Image

PROJECT_ID = 'forestwatch-ai-2026'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# Small pilot area in the Western Ghats; not a final forest boundary.
pilot_aoi = ee.Geometry.Rectangle([74.80, 13.00, 75.00, 13.20])

def mask_s2_clouds(image):
    scl = image.select('SCL')
    # Remove cloud shadow, medium/high cloud, cirrus and snow classes.
    clear = (scl.neq(3)
             .And(scl.neq(8))
             .And(scl.neq(9))
             .And(scl.neq(10))
             .And(scl.neq(11)))
    return image.updateMask(clear).select(['B4', 'B8']).copyProperties(image, ['system:time_start'])

def make_seasonal_ndvi(start_date, end_date):
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(pilot_aoi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
                  .map(mask_s2_clouds))
    count = collection.size().getInfo()
    if count == 0:
        raise RuntimeError(f'No Sentinel-2 scenes found for {start_date} to {end_date}.')
    composite = collection.median()
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi, count

print('Building season-matched NDVI composites...')
ndvi_2024, count_2024 = make_seasonal_ndvi('2024-01-01', '2024-04-01')
ndvi_2025, count_2025 = make_seasonal_ndvi('2025-01-01', '2025-04-01')
ndvi_change = ndvi_2025.subtract(ndvi_2024).rename('NDVI_change').clip(pilot_aoi)

# Conservative exploratory screen: previously greener pixels with a large decline.
# Threshold is a screening parameter, not a validated legal/deforestation rule.
candidate_pixels = ndvi_2024.gt(0.5).And(ndvi_change.lt(-0.20)).rename('candidate')
candidate_area_ha = (ee.Image.pixelArea().divide(10000)
                     .updateMask(candidate_pixels)
                     .rename('candidate_area_ha'))

mean_values = ee.Image.cat([ndvi_2024.rename('mean_NDVI_2024'),
                            ndvi_2025.rename('mean_NDVI_2025'),
                            ndvi_change.rename('mean_NDVI_change')]).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=pilot_aoi, scale=100,
    maxPixels=1e8, tileScale=4).getInfo()

area_values = candidate_area_ha.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=pilot_aoi, scale=100,
    maxPixels=1e8, tileScale=4).getInfo()

print('\n--- ForestWatch pilot screening summary ---')
print(f'Sentinel-2 scenes used, Jan–Mar 2024: {count_2024}')
print(f'Sentinel-2 scenes used, Jan–Mar 2025: {count_2025}')
print('Mean NDVI and mean change:', mean_values)
print('Exploratory candidate area (hectares):', area_values.get('candidate_area_ha', 0) or 0)
print('Interpretation: investigate candidate pixels; do not label them illegal deforestation.')

thumbnail_url = ndvi_change.getThumbURL({
    'region': pilot_aoi, 'dimensions': 700, 'format': 'png',
    'min': -0.4, 'max': 0.4,
    'palette': ['8c2d04', 'd8b365', 'f5f5f5', '5ab4ac', '01665e']
})
display(Image(url=thumbnail_url))

## Interpretation and limitations

- A negative NDVI change can result from harvest, fire, drought, seasonal variation, cloud contamination or other causes.
- The Jan–Mar windows are season-matched but do not eliminate all weather or phenology differences.
- The 0.5 baseline and −0.20 change thresholds are exploratory settings and must be tested against reference labels before any performance claim.
- The reported area is a coarse screening estimate at 100 m reduction scale, not a survey-grade boundary.
- Next we will add data-quality checks, alternative explanations and independent reference validation before calling any area a high-priority case.

## What happens next?

Once this access check succeeds, we will add cloud and shadow screening, season-matched composites, NDVI summaries and a transparent change-evidence table. We will validate each step before adding it to the dashboard.

## 4. Turn noisy changed pixels into candidate regions

This step groups adjacent candidate pixels into connected regions, removes very small clusters, and ranks the remaining regions by mapped area. It uses the variables created by Section 3, so run the notebook from the top in order. The minimum cluster size is a technical noise filter, not a scientific or legal threshold.

In [ ]:
import ee
from IPython.display import display, Image

# Check that the previous satellite-analysis section has been run.
required_names = ['pilot_aoi', 'ndvi_change', 'candidate_pixels']
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError('Please run Sections 1–3 from the top first. Missing: ' + ', '.join(missing_names))

# Count neighbouring candidate pixels. Eight-way connectivity includes diagonals.
connected_count = candidate_pixels.selfMask().connectedPixelCount(
    maxSize=100, eightConnected=True
)

# Keep connected groups containing at least 8 native-resolution pixels.
# At Sentinel-2 10 m resolution, 8 pixels represent about 0.08 ha before geometry effects.
clean_candidate_mask = (candidate_pixels
                        .updateMask(connected_count.gte(8))
                        .selfMask()
                        .toInt()
                        .rename('cluster'))

# Vectorize connected groups and summarize the NDVI change within each group.
vector_input = clean_candidate_mask.addBands(ndvi_change.rename('ndvi_change'))
candidate_regions = vector_input.reduceToVectors(
    reducer=ee.Reducer.mean(),
    geometry=pilot_aoi,
    scale=10,
    geometryType='polygon',
    eightConnected=True,
    labelProperty='cluster_id',
    maxPixels=100000000,
    tileScale=4
)

def add_region_metrics(feature):
    area_ha = feature.geometry().area(maxError=1).divide(10000)
    center = feature.geometry().centroid(maxError=1).coordinates()
    return feature.set({
        'area_ha': area_ha,
        'longitude': center.get(0),
        'latitude': center.get(1),
        'mean_ndvi_change': feature.get('mean')
    })

regions_with_metrics = candidate_regions.map(add_region_metrics)
# Exclude mapped fragments smaller than 0.1 ha, then show the 20 largest.
review_regions = (regions_with_metrics
                  .filter(ee.Filter.gte('area_ha', 0.1))
                  .sort('area_ha', False))

region_count = review_regions.size().getInfo()
top_regions = review_regions.limit(20).getInfo()

print('--- ForestWatch candidate-region summary ---')
print('Connected candidate regions >= 0.1 ha:', region_count)
print('Showing up to 20 largest regions:')
for index, feature in enumerate(top_regions.get('features', []), start=1):
    props = feature.get('properties', {})
    print(
        f"{index}. Area: {props.get('area_ha', float('nan')):.2f} ha | "
        f"Centre: {props.get('latitude', float('nan')):.5f}, "
        f"{props.get('longitude', float('nan')):.5f} | "
        f"Mean NDVI change: {props.get('mean_ndvi_change', float('nan')):.3f}"
    )

# Render the grouped regions as a simple image for visual inspection.
region_outline = ee.Image().byte().paint(
    featureCollection=review_regions,
    color=1,
    width=2
).clip(pilot_aoi)
region_thumb = region_outline.getThumbURL({
    'region': pilot_aoi,
    'dimensions': 700,
    'format': 'png',
    'min': 0,
    'max': 1,
    'palette': ['ffffff', 'b2182b']
})
display(Image(url=region_thumb))

print('\nReminder: these are vegetation-change candidates for human review, not confirmed deforestation.')

## Candidate-region interpretation

Connected-region grouping reduces isolated speckle but does not establish the cause of a change. Region area and centroid are approximate map summaries. Before assigning investigation priority, we must inspect image quality, compare radar evidence, check fire and other disturbance sources where possible, and validate against independent reference data.

## 5. View candidate regions over real satellite imagery

The previous red-on-white image was an outline-only diagnostic layer. This section places those outlines over a true-colour Sentinel-2 composite so the candidate shapes can be inspected in geographic context.

In [ ]:
import ee
from IPython.display import display, Image

required_names = ['pilot_aoi', 'review_regions']
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError('Please run Sections 1–4 from the top first. Missing: ' + ', '.join(missing_names))

def mask_s2_rgb(image):
    scl = image.select('SCL')
    clear = (scl.neq(3)
             .And(scl.neq(8))
             .And(scl.neq(9))
             .And(scl.neq(10))
             .And(scl.neq(11)))
    return image.updateMask(clear).select(['B4', 'B3', 'B2'])

rgb_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(pilot_aoi)
                  .filterDate('2025-01-01', '2025-04-01')
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
                  .map(mask_s2_rgb))

rgb_count = rgb_collection.size().getInfo()
if rgb_count == 0:
    raise RuntimeError('No Sentinel-2 RGB scenes are available for the 2025 pilot period.')

rgb_composite = rgb_collection.median().clip(pilot_aoi)
rgb_visual = rgb_composite.visualize(
    bands=['B4', 'B3', 'B2'], min=0, max=3000, gamma=1.2
)

outline = ee.Image().byte().paint(
    featureCollection=review_regions,
    color=1,
    width=2
).selfMask().clip(pilot_aoi)
outline_visual = outline.visualize(palette=['ff2d2d'], opacity=1)
overlay = rgb_visual.blend(outline_visual)

overlay_url = overlay.getThumbURL({
    'region': pilot_aoi,
    'dimensions': 900,
    'format': 'png'
})

print(f'Sentinel-2 scenes used for background composite: {rgb_count}')
print('Red outlines = candidate vegetation-change regions; inspect them against the underlying imagery.')
display(Image(url=overlay_url))

## 6. Add Sentinel-1 radar evidence

This section compares Sentinel-1 VV and VH radar backscatter for the same January–March periods in 2024 and 2025. It uses one orbit direction to reduce geometry differences. Radar changes are supporting evidence only: soil moisture, vegetation structure, terrain and acquisition geometry can all affect backscatter. This is not a validated deforestation classifier.

Run Sections 1–5 first in the same runtime.

In [ ]:
import ee

required_names = ['pilot_aoi', 'candidate_pixels', 'ndvi_change']
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError('Please run Sections 1–5 from the top first. Missing: ' + ', '.join(missing_names))

def get_s1_collection(start_date, end_date):
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(pilot_aoi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select(['VV', 'VH']))

s1_2024 = get_s1_collection('2024-01-01', '2024-04-01')
s1_2025 = get_s1_collection('2025-01-01', '2025-04-01')
count_s1_2024 = s1_2024.size().getInfo()
count_s1_2025 = s1_2025.size().getInfo()

print('Sentinel-1 ascending-orbit scenes, Jan–Mar 2024:', count_s1_2024)
print('Sentinel-1 ascending-orbit scenes, Jan–Mar 2025:', count_s1_2025)
if count_s1_2024 == 0 or count_s1_2025 == 0:
    raise RuntimeError('Sentinel-1 scenes are missing for one of the comparison periods. Do not interpret radar evidence; we will adjust the acquisition filter.')

# Sentinel-1 GRD VV/VH values in Earth Engine are in dB.
radar_2024 = s1_2024.median().clip(pilot_aoi)
radar_2025 = s1_2025.median().clip(pilot_aoi)
vv_change = radar_2025.select('VV').subtract(radar_2024.select('VV')).rename('VV_change_dB')
vh_change = radar_2025.select('VH').subtract(radar_2024.select('VH')).rename('VH_change_dB')

# A conservative exploratory agreement screen: an optical candidate pixel
# with a VH backscatter decrease of at least 1 dB. This is not a validated threshold.
radar_support = vh_change.lt(-1.0).rename('radar_support')
optical_and_radar = candidate_pixels.And(radar_support).rename('optical_radar_overlap')

summary = ee.Image.cat([
    vv_change,
    vh_change,
    optical_and_radar
]).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=pilot_aoi,
    scale=100,
    maxPixels=100000000,
    tileScale=4
).getInfo()

overlap_area = (ee.Image.pixelArea().divide(10000)
                .updateMask(optical_and_radar)
                .rename('overlap_ha')
                .reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=pilot_aoi,
                    scale=100,
                    maxPixels=100000000,
                    tileScale=4
                ).getInfo())

print('\n--- Sentinel-1 supporting-evidence summary ---')
print('Mean VV change (dB):', summary.get('VV_change_dB'))
print('Mean VH change (dB):', summary.get('VH_change_dB'))
print('Mean optical/radar overlap fraction at 100 m sampling:', summary.get('optical_radar_overlap'))
print('Exploratory optical + radar overlap area (ha):', overlap_area.get('overlap_ha', 0) or 0)
print('Interpretation: overlap is a review signal only; no cause or legality is inferred.')

radar_thumb = vh_change.getThumbURL({
    'region': pilot_aoi,
    'dimensions': 800,
    'format': 'png',
    'min': -3,
    'max': 3,
    'palette': ['8c2d04', 'f7f7f7', '2166ac']
})
from IPython.display import Image, display
display(Image(url=radar_thumb))

## Sentinel-1 interpretation and safeguards

- Radar backscatter changes are not uniquely associated with forest clearing.
- Soil moisture, rainfall, vegetation structure, slope/aspect and acquisition geometry can change VV/VH values.
- The −1 dB VH threshold is an exploratory screening parameter, not a literature-validated decision boundary for this site.
- Do not call optical/radar overlap a confirmed deforestation event. Inspect dates, imagery, topography, fire/landslide context and independent reference labels.
- Before reporting performance, create a spatially independent labelled validation sample and report precision, recall and uncertainty.